# Domain adaptation on digit images

**Goal:** train a digit classifier on one dataset (MNIST) and make it work well on a *different* one
(USPS) — **without using any USPS labels**. That is the problem *domain adaptation* solves: the two
datasets show the same ten digits, but the images look different, so a model trained on one usually
stumbles on the other.

In this tutorial you will:

1. load MNIST (the **source**, with labels) and USPS (the **target**, labels hidden);
2. build a small network with a **DANN** adapter (Domain-Adversarial Neural Network);
3. train it, and check how well it reads the unlabelled USPS digits.

It runs top-to-bottom in a few minutes on a free CPU — just press **Run in Colab** and go.
This is a lightweight teaching version; the full experiment lives next to this notebook in `main.py`.

## 1. Set up PyKale

On Google Colab, the next cell installs PyKale. Everywhere else it does nothing.

In [ ]:
# On Google Colab this installs PyKale (with its image extras). It is a no-op elsewhere.
import importlib.util
import sys

if "google.colab" in sys.modules and importlib.util.find_spec("kale") is None:
    !pip install --quiet "pykale[image]" matplotlib
    print("PyKale installed. If an import fails, use Runtime > Restart session, then run this again.")

## 2. Imports

Everything comes from PyKale (`kale.*`) and PyTorch Lightning — nothing from other files, so this
notebook is fully self-contained.

In [ ]:
import pytorch_lightning as pl

import kale.pipeline.domain_adapter as domain_adapter
from kale.embed.image_cnn import SmallCNNFeature
from kale.loaddata.image_access import DigitDataset
from kale.loaddata.multi_domain import MultiDomainAccess, MultiDomainDataset
from kale.predict.class_domain_nets import ClassNetSmallImage, DomainNetSmallImage
from kale.utils.seed import set_seed

## 3. Settings

A handful of plain values instead of a config file, so you can see (and tweak) everything at a glance.
The epoch count is deliberately small for a quick run — raise `MAX_EPOCHS` for better accuracy.

In [ ]:
SOURCE = "mnist"      # labelled dataset we train on
TARGET = "usps"       # unlabelled dataset we want to do well on
NUM_CLASSES = 10      # digits 0-9
DATA_ROOT = "./data"  # datasets download here on first run (MNIST ~12 MB, USPS ~2 MB)

SEED = 2025
MAX_EPOCHS = 3        # small for a quick CPU run; the full example trains far longer
BATCH_SIZE = 150
BASE_LR = 0.001

set_seed(SEED)

## 4. Load the data

MNIST is the **source** (we use its labels); USPS is the **target** (we pretend we have no labels).
`MultiDomainDataset` serves images from both so the adapter can learn what is *common* to them.

In [ ]:
source_data = DigitDataset(SOURCE.upper())
target_data = DigitDataset(TARGET.upper())

# Grayscale MNIST and USPS are 1-channel; this keeps the network's first layer right.
num_channels = max(
    DigitDataset.get_channel_numbers(source_data),
    DigitDataset.get_channel_numbers(target_data),
)

data_access = MultiDomainAccess(
    {
        SOURCE.upper(): DigitDataset.get_access(source_data, DATA_ROOT)[0],
        TARGET.upper(): DigitDataset.get_access(target_data, DATA_ROOT)[0],
    },
    NUM_CLASSES,
    return_domain_label=True,
)
dataset = MultiDomainDataset(data_access)
print(f"Loaded {SOURCE.upper()} (source) and {TARGET.upper()} (target).")

## 5. Build the model

Three small pieces:

- a **feature extractor** (`SmallCNNFeature`) that turns an image into a vector;
- a **classifier** that reads the digit from that vector;
- a **domain critic** that tries to tell *which dataset* an image came from.

**DANN** trains the feature extractor to fool the critic — so the features it learns look the same for
MNIST and USPS. The classifier then works on both, even though it only ever saw MNIST labels.

In [ ]:
feature_extractor = SmallCNNFeature(num_channels)
feature_dim = feature_extractor.output_size()

task_classifier = ClassNetSmallImage(feature_dim, NUM_CLASSES)
domain_critic = DomainNetSmallImage(feature_dim)

train_params = {
    "adapt_lambda": True,
    "adapt_lr": True,
    "lambda_init": 1.0,
    "nb_adapt_epochs": MAX_EPOCHS,
    "nb_init_epochs": 0,
    "init_lr": BASE_LR,
    "batch_size": BATCH_SIZE,
    "num_workers": 0,
    "optimizer": {
        "type": "SGD",
        "optim_params": {"momentum": 0.9, "weight_decay": 0.0005, "nesterov": True},
    },
}

model = domain_adapter.create_dann_like(
    method=domain_adapter.Method("DANN"),
    dataset=dataset,
    feature_extractor=feature_extractor,
    task_classifier=task_classifier,
    critic=domain_critic,
    target_domain=TARGET.upper(),
    **train_params,
)
print("Built a DANN model.")

## 6. Train

The classifier learns digits from MNIST while the adapter quietly pulls the MNIST and USPS features
together. We train on a fraction of the data for speed — drop `limit_train_batches` for the full run.

In [ ]:
trainer = pl.Trainer(
    max_epochs=MAX_EPOCHS,
    accelerator="cpu",
    devices=1,
    limit_train_batches=0.2,   # a fifth of each epoch -> a few minutes on CPU
    limit_val_batches=0.2,
    logger=False,
    enable_checkpointing=False,
)
trainer.fit(model)

## 7. How well does it read USPS?

The score below is accuracy on the **target** (USPS) digits — the ones the model never saw labels for.
Even this short run should beat a model trained on MNIST alone; more epochs push it higher.

In [ ]:
results = trainer.test(model)
results

## What next

- **Train for real.** Run the full example beside this notebook:
  `python main.py --cfg configs/MN2UP-DANN.yaml --devices 1`
- **Try other methods.** This example also ships CDAN, DAN, JAN, WDGRL and MME — swap the config in
  `configs/` and compare.
- **Use your own data.** The same recipe — feature extractor + classifier + domain critic — adapts
  between any two image sets that share the same labels.

PyKale docs: https://pykale.readthedocs.io · Library: https://github.com/pykale/pykale